# ICD-10 Category Prediction

Hybrid approach: rule-based shortcuts + TF-IDF / LinearSVC classifier.

The task is to assign the **first character** of a ICD-10 code to short clinical literals written by hospital staff. The literals are noisy: abbreviations, Catalan mixed with Spanish, occasional bare ICD codes; so a preprocessing step does most of the heavy lifting before anything learns.

## 1. Imports

In [ ]:
import os
import re
import unicodedata

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
import gc
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
import numpy as np

## 2. Data

Three sources are used for training:

- **`training_codification_data.csv`** - approx 13.7k labeled Kaggle pairs, same distribution as the leaderboard
- **`icd_d_p_pairs.csv`** - approx 180k official ICD-10 description/code pairs; adds vocabulary breadth but shifts the class distribution toward procedures
- **CodiEsp** (optional, [zenodo.org/records/3837305](https://zenodo.org/records/3837305)) - approx 8k real Spanish clinical mentions; worth aprox +1.6pp on local CV if available

In [4]:
KAGGLE_DIR = '/kaggle/input/competitions/uab-asho-ai-codification'
LOCAL_DIR  = './data/raw'


if os.path.isdir(KAGGLE_DIR):
    base, train_file, test_file = KAGGLE_DIR, 'codification_data.csv', 'leaderboard_data.csv'
else:
    base, train_file, test_file = LOCAL_DIR, 'training_codification_data.csv', 'test_leaderboard_data.csv'

train_df = pd.read_csv(f'{base}/{train_file}')
icd_df   = pd.read_csv(f'{base}/icd_d_p_pairs.csv')
test_df  = pd.read_csv(f'{base}/{test_file}')

codiesp = None
for cand in [
    './data/codiesp/final_dataset_v4_to_publish',
    '/kaggle/input/codiesp/final_dataset_v4_to_publish',
    '/kaggle/input/codiesp',
    '/kaggle/input/datasets/andreiadrianrgman/codiesp/codiesp/final_dataset_v4_to_publish'
]:
    if os.path.isfile(f'{cand}/train/trainX.tsv'):
        cols = ['articleID', 'label', 'Code', 'Literal', 'pos']
        codiesp = pd.concat([
            pd.read_csv(f'{cand}/train/trainX.tsv', sep='\t', header=None, names=cols),
            pd.read_csv(f'{cand}/dev/devX.tsv',     sep='\t', header=None, names=cols),
            pd.read_csv(f'{cand}/test/testX.tsv',   sep='\t', header=None, names=cols),
        ], ignore_index=True)
        codiesp['Code'] = codiesp['Code'].astype(str).str.upper().str.replace('.', '', regex=False)
        codiesp = codiesp[['Literal', 'Code']].drop_duplicates().reset_index(drop=True)
        print(f'CodiEsp loaded from {cand}: {len(codiesp):,} pairs')
        break

if codiesp is None:
    print('CodiEsp not found, running without it.')

print('train:', train_df.shape)
print('icd:', icd_df.shape)
print('test:', test_df.shape)
test_df.head()

CodiEsp not found, running without it.
train: (13700, 2)
icd: (179742, 3)
test: (6667, 2)


,id,Literal
0,1,AMNIODRENAJE
1,2,Hiperparatiroidismo primario
2,3,MIGRANYA parto
3,4,VHC
4,5,Absceso mama izq


## 3. Preprocessing

The literals are short and messy. Three normalisation steps run in sequence:

1. **Catalan to Spanish** - the hospital data mixes both languages; a small hand-built dictionary covers the most common terms

2. **Abbreviation expansion** - clinical shorthand (`HTA`, `GEA`, `EPOC`, ...) is replaced with its full Spanish form so the vectoriser sees real words; the dictionary was built from the training literals using the SEDOM medical abbreviation dictionary as reference

3. **Accent stripping + lowercase** - standard normalisation

The `extract_icd_from_literal` function handles the case where a doctor wrote the code itself instead of a description. It recognises both full ICD-10 codes and ICD-9 numeric codes (mapping the latter to the nearest ICD-10 chapter letter).

In [ ]:
# Abbreviation dictionary built from corpus analysis + SEDOM (http://diccionario.sedom.es/)
ABBREVIATIONS = {
    'HTA': 'hipertension arterial', 'IAM': 'infarto agudo miocardio',
    'FA': 'fibrilacion auricular', 'IC': 'insuficiencia cardiaca',
    'ICC': 'insuficiencia cardiaca congestiva', 'TEP': 'tromboembolismo pulmonar',
    'TVP': 'trombosis venosa profunda', 'SCA': 'sindrome coronario agudo',
    'BAV': 'bloqueo auriculoventricular', 'WPW': 'wolff parkinson white',
    'DAI': 'desfibrilador automatico implantable', 'CDI': 'cardioversor desfibrilador implantable',
    'DAP': 'ductus arterioso persistente', 'CIA': 'comunicacion interauricular',
    'CIV': 'comunicacion interventricular', 'HTP': 'hipertension pulmonar',
    'HTAP': 'hipertension arterial pulmonar', 'FOP': 'foramen oval permeable',
    'ACFA': 'arritmia completa por fibrilacion auricular',
    'ACXFA': 'arritmia completa por fibrilacion auricular',
    'TPSV': 'taquicardia paroxistica supraventricular',
    'TSV': 'taquicardia supraventricular', 'TV': 'taquicardia ventricular',
    'ESV': 'extrasistolia ventricular', 'INR': 'international normalized ratio',
    'ECG': 'electrocardiograma', 'EKG': 'electrocardiograma',
    'BRD': 'bloqueo rama derecha', 'BRDHH': 'bloqueo rama derecha haz his',
    'BRIHH': 'bloqueo rama izquierda haz his',
    'EPOC': 'enfermedad pulmonar obstructiva cronica',
    'SAHS': 'sindrome apnea hipopnea sueno', 'SAOS': 'sindrome apnea obstructiva sueno',
    'SAOHS': 'sindrome apnea obstructiva hipopnea sueno',
    'CPAP': 'continuous positive airway pressure', 'BIPAP': 'bilevel positive airway pressure',
    'VMNI': 'ventilacion mecanica no invasiva', 'VNI': 'ventilacion no invasiva',
    'TBC': 'tuberculosis', 'CNAF': 'canulas nasales alto flujo',
    'FPI': 'fibrosis pulmonar idiopatica', 'IRVS': 'infeccion respiratoria via superior',
    'DM': 'diabetes mellitus', 'DM1': 'diabetes mellitus tipo 1',
    'DM2': 'diabetes mellitus tipo 2', 'DMT2': 'diabetes mellitus tipo 2',
    'DMII': 'diabetes mellitus tipo 2', 'DMTII': 'diabetes mellitus tipo 2',
    'DMNID': 'diabetes mellitus no insulinodependiente',
    'DMG': 'diabetes mellitus gestacional', 'HLD': 'hiperlipidemia', 'DLP': 'dislipemia',
    'IMC': 'indice masa corporal', 'HPT': 'hiperparatiroidismo',
    'DMAE': 'degeneracion macular asociada edad',
    'IRC': 'insuficiencia renal cronica', 'IRA': 'insuficiencia renal aguda',
    'ERC': 'enfermedad renal cronica', 'ITU': 'infeccion tracto urinario',
    'ITUS': 'infeccion tracto urinario superior', 'RAO': 'retencion aguda orina',
    'IUE': 'incontinencia urinaria esfuerzo', 'IUU': 'incontinencia urinaria urgencia',
    'IUM': 'incontinencia urinaria mixta', 'RVU': 'reflujo vesicoureteral',
    'PNA': 'pielonefritis aguda',
    'ACV': 'accidente cerebrovascular', 'ACVA': 'accidente cerebrovascular agudo',
    'AVC': 'accidente vascular cerebral', 'AIT': 'accidente isquemico transitorio',
    'TCE': 'traumatismo craneoencefalico', 'LCR': 'liquido cefalorraquideo',
    'SNC': 'sistema nervioso central', 'EEG': 'electroencefalograma',
    'EMG': 'electromiograma', 'TDAH': 'trastorno deficit atencion hiperactividad',
    'TEA': 'trastorno espectro autista', 'TOC': 'trastorno obsesivo compulsivo',
    'TLP': 'trastorno limite personalidad', 'SGB': 'sindrome guillain barre',
    'STC': 'sindrome tunel carpiano', 'EMRR': 'esclerosis multiple remitente recurrente',
    'ERGE': 'enfermedad reflujo gastroesofagico', 'RGE': 'reflujo gastroesofagico',
    'HDA': 'hemorragia digestiva alta', 'HDB': 'hemorragia digestiva baja',
    'EII': 'enfermedad inflamatoria intestinal', 'NASH': 'non alcoholic steatohepatitis',
    'CPRE': 'colangiopancreatografia retrograda endoscopica',
    'SNG': 'sonda nasogastrica', 'GEA': 'gastroenteritis aguda', 'OH': 'alcohol etanol',
    'VIH': 'virus inmunodeficiencia humana', 'HIV': 'human immunodeficiency virus',
    'VHB': 'virus hepatitis B', 'VHC': 'virus hepatitis C', 'VHA': 'virus hepatitis A',
    'VPH': 'virus papiloma humano', 'HPV': 'human papillomavirus',
    'VRS': 'virus respiratorio sincitial', 'CMV': 'citomegalovirus',
    'PCR': 'parada cardiorrespiratoria', 'SARS': 'severe acute respiratory syndrome',
    'MRSA': 'methicillin resistant staphylococcus aureus',
    'SARM': 'staphylococcus aureus resistente meticilina',
    'BGN': 'bacilo gram negativo', 'ATB': 'antibiotico', 'COVID': 'coronavirus disease',
    'CA': 'cancer', 'NEO': 'neoplasia', 'ADK': 'adenocarcinoma',
    'QT': 'quimioterapia', 'QMT': 'quimioterapia', 'RT': 'radioterapia',
    'MTX': 'metotrexato', 'PSA': 'prostatic specific antigen',
    'LMA': 'leucemia mieloide aguda', 'LLC': 'leucemia linfocitica cronica',
    'LLA': 'leucemia linfoblastica aguda', 'LNH': 'linfoma no hodgkin',
    'SMD': 'sindrome mielodisplasico', 'PTI': 'purpura trombocitopenica idiopatica',
    'BSGC': 'biopsia selectiva ganglio centinela', 'LOE': 'lesion ocupante espacio',
    'CIN': 'cervical intraepithelial neoplasia', 'VIN': 'vulvar intraepithelial neoplasia',
    'BRCA': 'breast cancer gene', 'BRCA1': 'breast cancer gene 1',
    'BRCA2': 'breast cancer gene 2', 'TNE': 'tumor neuroendocrino',
    'RPM': 'rotura prematura membranas', 'RCIU': 'retraso crecimiento intrauterino',
    'GEU': 'gestacion ectopica uterina', 'SOP': 'sindrome ovario poliquistico',
    'FIV': 'fecundacion in vitro', 'DIU': 'dispositivo intrauterino',
    'NST': 'non stress test', 'RCTG': 'registro cardiotocografico',
    'ILE': 'interrupcion legal embarazo', 'IVE': 'interrupcion voluntaria embarazo',
    'HPP': 'hemorragia postparto', 'HELLP': 'hemolysis elevated liver enzymes low platelets',
    'DPPNI': 'desprendimiento prematuro placenta normalmente inserta',
    'SG': 'semanas gestacion', 'RN': 'recien nacido', 'RNPT': 'recien nacido prematuro',
    'RPBF': 'riesgo perdida bienestar fetal', 'SFA': 'sufrimiento fetal agudo',
    'EHE': 'enfermedad hipertensiva embarazo', 'APGAR': 'apgar score',
    'LES': 'lupus eritematoso sistemico', 'AR': 'artritis reumatoide',
    'AINES': 'antiinflamatorios no esteroideos', 'PTR': 'protesis total rodilla',
    'PTC': 'protesis total cadera', 'AAS': 'acido acetilsalicilico',
    'HBPM': 'heparina bajo peso molecular', 'LCA': 'ligamento cruzado anterior',
    'TAC': 'tomografia axial computarizada', 'TC': 'tomografia computarizada',
    'RM': 'resonancia magnetica', 'RMN': 'resonancia magnetica nuclear',
    'ECO': 'ecografia', 'RX': 'radiografia', 'PET': 'tomografia emision positrones',
    'PAAF': 'puncion aspiracion aguja fina', 'BAG': 'biopsia aguja gruesa',
    'IOT': 'intubacion orotraqueal', 'NPT': 'nutricion parenteral total',
    'CVC': 'cateter venoso central', 'RTU': 'reseccion transuretral',
    'LAP': 'laparoscopia', 'IQ': 'intervencion quirurgica',
    'ETT': 'ecocardiografia transtoraxica', 'ECOTV': 'ecografia transvaginal',
    'EDA': 'endoscopia digestiva alta', 'FACO': 'facoemulsificacion',
    'PEG': 'pequeno para edad gestacional', 'GEG': 'grande para edad gestacional',
    'APP': 'amenaza parto prematuro', 'APLV': 'alergia proteina leche vaca',
    'IPLV': 'intolerancia proteina leche vaca',
    'SD': 'sindrome', 'SF': 'sindrome febril', 'IZQ': 'izquierdo',
    'IZDA': 'izquierda', 'DCHA': 'derecha', 'EEII': 'extremidades inferiores',
    'ABD': 'abdomen', 'OD': 'ojo derecho', 'OI': 'ojo izquierdo',
    'ADVP': 'adicto drogas via parenteral',
    'SIADH': 'sindrome inadecuada secrecion hormona antidiuretica',
    'SAF': 'sindrome antifosfolipido', 'GGT': 'gamma glutamil transferasa',
    'TPH': 'trasplante progenitores hematopoyeticos',
    'EICH': 'enfermedad injerto contra huesped', 'POP': 'prolapso organos pelvicos',
    'SUA': 'sangrado uterino anormal', 'TCA': 'trastorno conducta alimentaria',
    'RCP': 'reanimacion cardiopulmonar', 'GSRH': 'grupo sanguineo rh',
    'RH': 'factor rhesus', 'LUES': 'sifilis', 'PTH': 'paratohormona',
    'ANTI-D': 'inmunoglobulina anti D',
}

CATALAN_TO_SPANISH = {
    'migranya':      'migranya',
    'cos estrany':   'cuerpo extrano',
    'cos':           'cuerpo',
    'estrany':       'extrano',
    'part':          'parto',
    'malaltia':      'enfermedad',
    'infeccio':      'infeccion',
    'embaras':       'embarazo',
    'diabetis':      'diabetes',
    'hipertensio':   'hipertension',
    'pneumonia':     'neumonia',
    'limfoma':       'linfoma',
    'cataracta':     'catarata',
    'artrosi':       'artrosis',
    'osteoporosi':   'osteoporosis',
    'calcul':        'calculo',
    'dret':          'derecho',
    'dreta':         'derecha',
    'esquerra':      'izquierda',
    'agut':          'agudo',
    'cronic':        'cronico',
    'amniodrenatge': 'amniodrenaje',
    'laceracio':     'laceracion',
    'laceracions':   'laceraciones',
    'postpart':      'postparto',
    'cesaria':       'cesarea',
}

# ICD-9 numeric codes mapped to the nearest ICD-10 chapter
ICD9_CHAPTER_MAP = [
    ((1,   139), 'A'),
    ((140, 239), 'C'),
    ((240, 279), 'E'),
    ((280, 289), 'D'),
    ((290, 319), 'F'),
    ((320, 389), 'G'),
    ((390, 459), 'I'),
    ((460, 519), 'J'),
    ((520, 579), 'K'),
    ((580, 629), 'N'),
    ((630, 679), 'O'),
    ((680, 709), 'L'),
    ((710, 739), 'M'),
    ((740, 759), 'Q'),
    ((760, 779), 'P'),
    ((780, 799), 'R'),
    ((800, 999), 'S'),
]

ICD10_FULL   = re.compile(r'^[A-Z]\d{2}\.?\d*[A-Z0-9]*$')
ICD10_INLINE = re.compile(r'\b([A-Z]\d{2}\.?\d*[A-Z0-9]*)\b')
ICD9_V_E     = re.compile(r'^[VE]\d{2}\.?\d*$')
ICD9_NUMERIC = re.compile(r'^\d{3}\.?\d*$')


def icd9_to_category(code_str):
    try:
        num = int(str(code_str).strip().split('.')[0])
    except ValueError:
        return None
    for (lo, hi), cat in ICD9_CHAPTER_MAP:
        if lo <= num <= hi:
            return cat
    return None


def remove_accents(text):
    nfd = unicodedata.normalize('NFD', text)
    return unicodedata.normalize('NFC', ''.join(c for c in nfd if unicodedata.category(c) != 'Mn'))


def apply_catalan_dict(text):
    # longer keys first so 'cos estrany' is matched before 'cos'
    for cat, esp in sorted(CATALAN_TO_SPANISH.items(), key=lambda x: -len(x[0])):
        text = re.sub(r'\b' + re.escape(cat) + r'\b', esp, text, flags=re.IGNORECASE)
    return text


def apply_abbreviations(text):
    return ' '.join(ABBREVIATIONS.get(tok.upper(), tok) for tok in text.split())


def preprocess(text):
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.strip().lower()
    text = remove_accents(text)
    text = apply_catalan_dict(text)
    text = apply_abbreviations(text)
    return re.sub(r'\s+', ' ', text).strip()


def extract_icd_from_literal(literal):
    """If the literal is (or contains) an ICD code, return it directly.
    ICD-9 numerics are mapped to the nearest ICD-10 chapter letter.
    ICD-9 V/E supplementary codes are discarded (no clean mapping)."""
    if not isinstance(literal, str):
        return None
    text = literal.strip().upper()
    if ICD9_NUMERIC.match(text):
        return icd9_to_category(text)
    if ICD9_V_E.match(text):
        return None
    if ICD10_FULL.match(text):
        return text.replace('.', '')
    m = ICD10_INLINE.search(text)
    if m:
        candidate = m.group(1)
        if not ICD9_V_E.match(candidate):
            return candidate.replace('.', '')
    m9 = ICD9_NUMERIC.search(text)
    if m9:
        return icd9_to_category(m9.group())
    return None

## 4. Apply preprocessing

In [6]:
train_df['processed']  = train_df['Literal'].fillna('').apply(preprocess)
test_df['processed']   = test_df['Literal'].fillna('').apply(preprocess)
icd_df['processed']    = icd_df['Description'].fillna('').apply(preprocess)
icd_df = icd_df[icd_df['processed'] != ''].dropna(subset=['Code']).reset_index(drop=True)

if codiesp is not None:
    codiesp['processed'] = codiesp['Literal'].fillna('').apply(preprocess)
    codiesp = codiesp[codiesp['processed'] != ''].dropna(subset=['Code']).reset_index(drop=True)

train_df[['Literal', 'processed']].head(8)

,Literal,processed
0,Hiperreactividad bronquial,hiperreactividad bronquial
1,broncoespástica,broncoespastica
2,miocardiopatía dilatada,miocardiopatia dilatada
3,HTA irc 6,hipertension arterial insuficiencia renal cron...
4,Crisis febriles atípicas,crisis febriles atipicas
5,prótesis mama,protesis mama
6,hernia discal parto,hernia discal parto
7,shock septico,shock septico


In [7]:
# append the icd_df to the training data
train_df = pd.concat([train_df, icd_df.drop(columns=['D_P', 'Description'])], ignore_index=True)

## 5. Model

A helper function that builds and fits the full pipeline given any set of training rows. Keeping it in one place avoids duplicating code between the validation run and the final submission run.

**Feature representation**: a `FeatureUnion` of two TF-IDF vectorisers:
- word n-grams (1-2): captures semantics and common multi-word terms
- character n-grams (3-5, `char_wb`): captures medical morphology like *-itis*, *-ectomía*, *-plasía* regardless of stemming

**Classifier**: `LinearSVC` with `C=0.7`, tuned on a 20% held-out split of the training data. `class_weight='balanced'` was tested but hurts performance: `icd_df` is procedure-heavy (codes 0-9) while the leaderboard distribution is dominated by O and Z. Weighting would amplify the wrong classes.

**Prediction priority**:
1. If the literal is itself an ICD code, use it directly (or map ICD-9 to chapter letter)
2. LinearSVC prediction
3. Cosine-similarity fallback against `icd_df` (rarely triggered)

In [ ]:


# Build dictionary from full code to D/P
code_to_dp = dict(zip(icd_df['Code'].astype(str), icd_df['D_P']))

def get_group(code):
    """Look up full code in dictionary. Default to procedure if not found."""
    return 'disease' if code_to_dp.get(str(code), 'P') == 'D' else 'procedure'

def build_hierarchical_pipeline(tr, icd, extra=None, C=1, icd_sample=1):
    icd = icd.sample(frac=icd_sample, random_state=42)

    tr_X  = tr['processed'].fillna('').values
    tr_y  = tr['Code'].astype(str).values
    icd_X = icd['processed'].values
    icd_y = icd['Code'].astype(str).values
    sources_X = [tr_X, icd_X]
    sources_y = [tr_y, icd_y]
    sources_w = [np.ones(len(tr_X)), np.full(len(icd_X), 0.2)]
    if extra is not None:
        extra_X = extra['processed'].values
        extra_y = extra['Code'].astype(str).values
        sources_X.append(extra_X)
        sources_y.append(extra_y)
        sources_w.append(np.ones(len(extra_X)))
    X = np.concatenate(sources_X)
    y = np.concatenate(sources_y)
    w = np.concatenate(sources_w)
    keep = (X != '') & (y != '')
    X, y, w = X[keep], y[keep], w[keep]

    vec = FeatureUnion([
        ('word', TfidfVectorizer(ngram_range=(1, 5), max_features=40000, sublinear_tf=True)),
        ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 7),
                                 max_features=80000, sublinear_tf=True, min_df=3)),
    ])
    X_vec = vec.fit_transform(X)

    # Level 1 - look up full code in dictionary
    y_l1 = np.array([get_group(c) for c in y])
    clf_l1 = LinearSVC(C=C, max_iter=1000)
    clf_l1.fit(X_vec, y_l1, sample_weight=w)

    # Level 2 targets are still first character 
    y_first = np.array([c[0] for c in y])

    disease_mask   = y_l1 == 'disease'
    procedure_mask = ~disease_mask
    X_disease      = X_vec[disease_mask]
    X_procedure    = X_vec[procedure_mask]
    del X_vec; gc.collect()

    # Level 2a - predict first char within disease group
    clf_disease = LinearSVC(C=C, max_iter=1000)
    clf_disease.fit(X_disease, y_first[disease_mask], sample_weight=w[disease_mask])
    del X_disease; gc.collect()

    # Level 2b - predict first char within procedure group
    clf_procedure = LinearSVC(C=C, max_iter=1000)
    clf_procedure.fit(X_procedure, y_first[procedure_mask], sample_weight=w[procedure_mask])
    del X_procedure; gc.collect()

    kb_vec = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)
    kb_mat = kb_vec.fit_transform(icd['processed'])

    print(f'  rows: {len(X):,} | disease: {disease_mask.sum():,} '
          f'| procedure: {procedure_mask.sum():,}')

    return vec, clf_l1, clf_disease, clf_procedure, kb_vec, kb_mat


def predict_hierarchical(df, vec, clf_l1, clf_disease, clf_procedure,
                          kb_vec, kb_mat, icd, batch_size=500):
    proc  = df['processed'].fillna('').values
    X_vec = vec.transform(proc)

    # Level 1: route each sample
    l1_pred = clf_l1.predict(X_vec)

    # Level 2: apply the right subclassifier per sample
    final_clf  = np.where(l1_pred == 'disease', 'disease', 'procedure')
    predictions = np.empty(len(proc), dtype=object)

    disease_mask   = final_clf == 'disease'
    procedure_mask = ~disease_mask

    if disease_mask.any():
        predictions[disease_mask]   = clf_disease.predict(X_vec[disease_mask])
    if procedure_mask.any():
        predictions[procedure_mask] = clf_procedure.predict(X_vec[procedure_mask])

    # cosine similarity in batches to avoid OOM
    sim_indices = np.empty(len(proc), dtype=int)
    kb_query    = kb_vec.transform(proc)

    for start in range(0, len(proc), batch_size):
        end   = min(start + batch_size, len(proc))
        batch = cosine_similarity(kb_query[start:end], kb_mat)
        sim_indices[start:end] = batch.argmax(axis=1)
        del batch

    sim_p  = icd['Code'].values[sim_indices]
    shorts = df['Literal'].apply(extract_icd_from_literal)

    out = []
    for sc, svc, sim in zip(shorts, predictions, sim_p):
        if isinstance(sc, str) and sc:
            out.append(sc[0])
        elif isinstance(svc, str) and svc:
            out.append(svc)
        else:
            out.append(sim[0] if isinstance(sim, str) and sim else 'null')
    return out

## 6. Validation (80/20 stratified split)

Stratified split so every category keeps its natural proportion in both halves. The model is refitted on the 80% only; `icd_df` is included in both runs as it contains no information about the hold-out labels. This is to see how the model performs on unseen data before committing to a submission.

In [ ]:
RUN_VALIDATION = True

if RUN_VALIDATION:
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    y_cat = train_df['Code'].astype(str).str[0]
    tr_idx, val_idx = next(sss.split(train_df, y_cat))

    tr_split  = train_df.iloc[tr_idx].reset_index(drop=True)
    val_split = train_df.iloc[val_idx].reset_index(drop=True)

    print('Fitting on 80% split...')
    vec_v, clf_l1_v, clf_disease_v, clf_procedure_v, kb_vec_v, kb_mat_v = \
        build_hierarchical_pipeline(tr_split, icd_df, codiesp)

    # Level 1 accuracy check
    # y_l1_true = ['disease' if c in DISEASE_CATS else 'procedure'
    #              for c in val_split['Code'].astype(str).str[0]]
    y_l1_true = val_split['Code'].astype(str).apply(get_group).tolist()
    y_l1_pred = clf_l1_v.predict(vec_v.transform(val_split['processed'].fillna('').values))
    print(f'Level 1 accuracy (disease/procedure split): {accuracy_score(y_l1_true, y_l1_pred):.4f}')

    # Full hierarchical prediction
    y_pred = predict_hierarchical(val_split, vec_v, clf_l1_v, clf_disease_v,
                                   clf_procedure_v, kb_vec_v, kb_mat_v, icd_df)
    y_true = val_split['Code'].astype(str).str[0].tolist()

    acc = accuracy_score(y_true, y_pred)
    print(f'Validation accuracy: {acc:.4f}  ({len(y_true)} samples)\n')
    print(classification_report(y_true, y_pred, zero_division=0))

    err = pd.DataFrame({'true': y_true, 'pred': y_pred})
    err = err[err['true'] != err['pred']]
    top_conf = (err.groupby(['true', 'pred']).size()
                .reset_index(name='n')
                .sort_values('n', ascending=False)
                .head(10))
    print('Top-10 confusions (true → predicted):')
    print(top_conf.to_string(index=False))

Fitting on 80% split...
  rows: 334,495 | disease: 188,546 | procedure: 145,949
Level 1 accuracy (disease/procedure split): 0.1641
Validation accuracy: 0.9667  (38689 samples)

              precision    recall  f1-score   support

           0       0.99      0.99      0.99     13833
           1       0.78      0.81      0.79       118
           2       0.88      0.75      0.81       253
           3       0.89      0.82      0.85       370
           4       0.82      0.65      0.73       164
           5       0.49      0.35      0.41        92
           6       0.56      0.21      0.31       137
           7       0.60      0.37      0.46        99
           8       0.74      0.51      0.60        97
           9       0.53      0.41      0.46        74
           A       0.89      0.94      0.92       143
           B       0.94      0.96      0.95       834
           C       0.97      0.96      0.96       428
           D       0.94      0.95      0.95       681
           E

## 7. Final model + submission

Retrain on **all** available labeled data (full `train_df` + `icd_df` + CodiEsp if present) before predicting on the leaderboard set.

In [26]:
print('Fitting on full training data...')
vec_f, clf_l1_f, clf_disease_f, clf_procedure_f, kb_vec_f, kb_mat_f = \
    build_hierarchical_pipeline(train_df, icd_df, codiesp)

test_df['y_category'] = predict_hierarchical(
    test_df, vec_f, clf_l1_f, clf_disease_f, clf_procedure_f,
    kb_vec_f, kb_mat_f, icd_df
)

submission = test_df[['id', 'Literal', 'y_category']].copy()

assert list(submission.columns) == ['id', 'Literal', 'y_category']
assert submission['id'].notna().all()
assert submission['Literal'].notna().all()
assert submission['y_category'].notna().all() and (submission['y_category'] != '').all()
assert len(submission) == len(test_df)

submission.to_csv('submission.csv', index=False)
print(f'\nWrote {len(submission)} predictions to submission.csv')
print()
print(submission.head(8).to_string())
print()
print(submission['y_category'].value_counts().to_string())

Fitting on full training data...
  rows: 373,184 | disease: 210,424 | procedure: 162,760

Wrote 6667 predictions to submission.csv

   id                       Literal y_category
0   1                  AMNIODRENAJE          1
1   2  Hiperparatiroidismo primario          E
2   3                MIGRANYA parto          O
3   4                           VHC          B
4   5              Absceso mama izq          N
5   6                  miomas parto          D
6   7                   cos estrany          E
7   8                           TCE          S

y_category
O    884
Z    823
0    633
N    322
B    315
E    278
K    256
I    246
R    232
3    220
D    212
F    186
M    169
V    156
J    153
8    142
G    140
6    137
4    122
C    114
9    114
1    110
5    104
2     97
Q     90
H     83
L     80
P     67
7     58
S     42
T     28
A     22
Y     12
W     10
U      8
X      2
